In [1]:
cfg = dict(
    seq_length  = 160,
    d_model     = 256,
    latent_dim  = 64,   # latent dimension
    n_head      = 8,
    enc_layers  = 7,
    dec_layers  = 7,
    ff_dim      = 1024,
    dropout     = 0.05,
    emb_dropout = 0.05,

    # special token indices (match your vocabulary)
    pad_idx     = 0,
    sos_idx     = 2,
    eos_idx     = 3,
    # -------- regularization tweaks --------
    label_smoothing = 0.02,   # 0.0 to disable
    corruption_p     = 0.05,  # word-dropout on decoder inputs (train only)

    # -------- validation / decoding --------
    beam_every  = 5,   # run beam metrics every N epochs
    beam_size   = 5)

In [2]:
from pathlib import Path

import torch, torch.nn as nn
import model_bs as mdl
import data_utils as du

# --- paths/config ---
PROJECT_ROOT = Path.cwd()
vocab_path = PROJECT_ROOT / "vocab.json"
ckpt_path = PROJECT_ROOT / "checkpoints" / "best_model.pth"
Dye_csv = PROJECT_ROOT / "Data" / "dye.csv"
test_csv = PROJECT_ROOT / "Data" / "Test.csv"
Test_pubchem_dir = PROJECT_ROOT / "Data" / "pubchem_test_data" / "inputs"

# --- load vocab ---
token_to_idx, idx_to_token = du.load_or_create_vocabulary(csv_paths=[], cache_path=vocab_path, test_smiles=None)
assert token_to_idx["<PAD>"] == cfg["pad_idx"]
assert token_to_idx["<SOS>"] == cfg["sos_idx"]
assert token_to_idx["<EOS>"] == cfg["eos_idx"]

# --- build the same architecture you trained ---
model = mdl.LSTM_VAE_Trans(
    vocab_size=len(token_to_idx),
    d_model=cfg["d_model"],
    latent_dim=cfg["latent_dim"],
    pad_idx=cfg["pad_idx"],
    sos_idx=cfg["sos_idx"],
    eos_idx=cfg["eos_idx"],
    enc_layers=cfg["enc_layers"],
    dec_layers=cfg["dec_layers"],
    nhead=cfg["n_head"],
    dropout=cfg["dropout"],
    max_len=cfg["seq_length"],
    dim_feedforward=cfg["ff_dim"])

# --- load weights robustly (handles 'module.' prefixes if any) ---
state = torch.load(ckpt_path, map_location="cpu")
try:
    model.load_state_dict(state, strict=True)
except RuntimeError:
    # remove a leading 'module.' if the checkpoint came from DataParallel
    from collections import OrderedDict
    new_state = OrderedDict()
    for k, v in state.items():
        new_state[k.replace("module.", "", 1)] = v
    model.load_state_dict(new_state, strict=True)

# --- device & optional DataParallel for speed (not required) ---
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device)
print("Trainable params:", du.count_parameters(model))
print(f"Encoder parameters: {du.count_parameters(model.encoder)}")
model.eval()

# If you want to keep everything single-GPU-friendly for beam_search, DON'T wrap in DataParallel.
# If you DO wrap, remember to pass model.module to functions that call custom methods.

[vocab] loaded cached vocabulary from /Users/md_halim_mondol/Desktop/JCC-Wiley_R1/LTVAE/vocab.json (69 tokens)
Trainable params: 10246597
Encoder parameters: 2786048


LSTM_VAE_Trans(
  (encoder): EncoderBiLSTM(
    (emb): Embedding(69, 256, padding_idx=0)
    (emb_ln): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (emb_do): Dropout(p=0.1, inplace=False)
    (lstm): LSTM(256, 128, num_layers=7, batch_first=True, dropout=0.05, bidirectional=True)
    (out_do): Dropout(p=0.05, inplace=False)
    (seq_ln): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (pool_ln): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (to_mu): Linear(in_features=256, out_features=64, bias=True)
  (to_logvar): Linear(in_features=256, out_features=64, bias=True)
  (latent_to_token): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): TransformerDecoder(
    (emb): Embedding(69, 256, padding_idx=0)
    (emb_ln): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (pe): PositionalEncoding(
      (dropout): Dropout(p=0.05, inplace=False)
    )

In [3]:
import pandas as pd

test_std_sources = [
    (csv_path.stem, csv_path)
    for csv_path in sorted(Test_pubchem_dir.glob("*.csv"))
]

def load_smiles_series(csv_path):
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    smiles_col = "smiles" if "smiles" in df.columns else next(c for c in df.columns if "smile" in c.lower())
    return df[smiles_col].dropna().astype(str)

std_rows = []
all_smiles = []

for dataset_name, csv_path in test_std_sources:
    smiles = load_smiles_series(csv_path)
    char_lengths = smiles.str.len()
    token_lengths = smiles.map(lambda s: len(du.tokenize_smiles(s)))
    all_smiles.append(smiles)

    std_rows.append({
        "dataset": dataset_name,
        "n_smiles": len(smiles),
        "mean_smiles_chars": char_lengths.mean(),
        "std_smiles_chars": char_lengths.std(),
        "mean_smiles_tokens": token_lengths.mean(),
        "std_smiles_tokens": token_lengths.std(),
    })

combined_smiles = pd.concat(all_smiles, ignore_index=True)
combined_char_lengths = combined_smiles.str.len()
combined_token_lengths = combined_smiles.map(lambda s: len(du.tokenize_smiles(s)))
std_rows.append({
    "dataset": "overall",
    "n_smiles": len(combined_smiles),
    "mean_smiles_chars": combined_char_lengths.mean(),
    "std_smiles_chars": combined_char_lengths.std(),
    "mean_smiles_tokens": combined_token_lengths.mean(),
    "std_smiles_tokens": combined_token_lengths.std(),
})

test_set_std_df = pd.DataFrame(std_rows)
per_dataset_df = test_set_std_df[test_set_std_df["dataset"] != "overall"]

among_test_sets_std_df = pd.DataFrame([{
    "summary": "standard deviation among test sets",
    "n_test_sets": len(per_dataset_df),
    "std_of_n_smiles": per_dataset_df["n_smiles"].std(),
    "std_of_mean_smiles_chars": per_dataset_df["mean_smiles_chars"].std(),
    "std_of_std_smiles_chars": per_dataset_df["std_smiles_chars"].std(),
    "std_of_mean_smiles_tokens": per_dataset_df["mean_smiles_tokens"].std(),
    "std_of_std_smiles_tokens": per_dataset_df["std_smiles_tokens"].std(),
}])

display(test_set_std_df.round(3))
display(among_test_sets_std_df.round(3))

,dataset,n_smiles,mean_smiles_chars,std_smiles_chars,mean_smiles_tokens,std_smiles_tokens
0,drug_like,4296,44.171,13.574,36.948,11.765
1,dyes_pigments,1728,46.311,13.503,38.795,10.332
2,natural_products,3862,39.126,10.880,33.472,9.556
3,polymers_monomers,3513,31.325,14.270,27.242,13.129
4,protein_enzyme_chemistry,2036,42.271,14.115,33.833,12.070
5,overall,15435,39.974,14.185,33.665,12.128


,summary,n_test_sets,std_of_n_smiles,std_of_mean_smiles_chars,std_of_std_smiles_chars,std_of_mean_smiles_tokens,std_of_std_smiles_tokens
0,standard deviation among test sets,5,1139.656,5.839,1.376,4.406,1.424


In [4]:
import pandas as pd
from inference import reconstruct_smiles_table

PUBCHEM_TEST_DIR = Test_pubchem_dir
OUTPUT_DIR = PROJECT_ROOT / "Data" / "pubchem_test_data" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 64
DRUG_LIKE_SAMPLE_N = 5000

test_files = [
    PUBCHEM_TEST_DIR / "drug_like.csv",
    PUBCHEM_TEST_DIR / "dyes_pigments.csv",
    PUBCHEM_TEST_DIR / "natural_products.csv",
    PUBCHEM_TEST_DIR / "polymers_monomers.csv",
    PUBCHEM_TEST_DIR / "protein_enzyme_chemistry.csv",
]

# Use the unwrapped model object for beam_search.
m = model  # if you ever wrap with DataParallel, use: model.module

def load_smiles_for_file(csv_path):
    nrows = DRUG_LIKE_SAMPLE_N if csv_path.name == "drug_like.csv" else None
    df = pd.read_csv(csv_path, nrows=nrows)
    col = "smiles" if "smiles" in df.columns else next(c for c in df.columns if "smile" in c.lower())
    return df[col].dropna().astype(str).tolist()

def token_accuracy_counts(gold, pred):
    g = du.tokenize_smiles(gold)
    p = du.tokenize_smiles(pred)
    total = min(len(g), len(p))
    if total == 0:
        return 0, 0
    correct = sum(gi == pi for gi, pi in zip(g[:total], p[:total]))
    return correct, total

def summarize_reconstruction(name, df_rec):
    tot_corr = tot_tok = 0
    for gold, pred in zip(df_rec["input"], df_rec["reconstructed"]):
        c, t = token_accuracy_counts(gold, pred)
        tot_corr += c
        tot_tok += t

    token_acc = tot_corr / tot_tok if tot_tok else float("nan")
    string_acc = (df_rec["input"] == df_rec["reconstructed"]).mean() if len(df_rec) else float("nan")
    validity_pct = 100.0 * (df_rec["valid"] == "yes").mean() if len(df_rec) else float("nan")
    avg_lev = df_rec["lev"].mean() if len(df_rec) else float("nan")

    return {
        "dataset": name,
        "n_smiles": len(df_rec),
        "overall_acc": token_acc,
        "string_level_acc": string_acc,
        "validity_percentage": validity_pct,
        "avg_lev_dist": avg_lev,
    }

summary_rows = []
all_reconstructions = []

for csv_path in test_files:
    dataset_name = csv_path.stem
    smiles = load_smiles_for_file(csv_path)
    print(f"Running {dataset_name}: {len(smiles)} SMILES with batch_size={BATCH_SIZE}")

    df_rec = reconstruct_smiles_table(
        smiles_list=smiles,
        test_csv=None,
        model=m,
        token_to_idx=token_to_idx,
        idx_to_token=idx_to_token,
        seq_length=cfg["seq_length"],
        pad_idx=cfg["pad_idx"],
        sos_idx=cfg["sos_idx"],
        eos_idx=cfg["eos_idx"],
        device=device,
        mode="beam",
        beam_size=cfg["beam_size"],
        batch_size=BATCH_SIZE,
        progress_every=25,
    )

    out_csv = OUTPUT_DIR / f"{dataset_name}_reconstruction_beam.csv"
    df_rec.to_csv(out_csv, index=False)
    print(f"Saved {out_csv}")

    summary = summarize_reconstruction(dataset_name, df_rec)
    summary_rows.append(summary)
    all_reconstructions.append(df_rec.assign(dataset=dataset_name))
    print(
        f"{dataset_name}: overall_acc={summary['overall_acc']:.4f}, "
        f"string_level_acc={summary['string_level_acc']:.4f}, "
        f"validity={summary['validity_percentage']:.2f}%, "
        f"avg_lev_dist={summary['avg_lev_dist']:.3f}"
    )

combined_df = pd.concat(all_reconstructions, ignore_index=True) if all_reconstructions else pd.DataFrame()
if len(combined_df):
    overall_summary = summarize_reconstruction("overall", combined_df)
    summary_rows.append(overall_summary)

summary_df = pd.DataFrame(summary_rows)
summary_csv = OUTPUT_DIR / "pubchem_reconstruction_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"Saved {summary_csv}")

display(summary_df)
if len(combined_df):
    display(combined_df[["dataset", "input", "reconstructed", "valid", "lev"]].head(10))

Running drug_like: 4296 SMILES with batch_size=64
decoded batch 1/68 (64/4296 SMILES)
decoded batch 25/68 (1600/4296 SMILES)
decoded batch 50/68 (3200/4296 SMILES)
decoded batch 68/68 (4296/4296 SMILES)
Saved /Users/md_halim_mondol/Desktop/JCC-Wiley_R1/LTVAE/Data/pubchem_test_data/outputs/drug_like_reconstruction_beam.csv
drug_like: overall_acc=0.8201, string_level_acc=0.4153, validity=77.35%, avg_lev_dist=4.360
Running dyes_pigments: 1728 SMILES with batch_size=64
decoded batch 1/27 (64/1728 SMILES)
decoded batch 25/27 (1600/1728 SMILES)
decoded batch 27/27 (1728/1728 SMILES)
Saved /Users/md_halim_mondol/Desktop/JCC-Wiley_R1/LTVAE/Data/pubchem_test_data/outputs/dyes_pigments_reconstruction_beam.csv
dyes_pigments: overall_acc=0.6589, string_level_acc=0.0394, validity=50.06%, avg_lev_dist=9.455
Running natural_products: 3862 SMILES with batch_size=64
decoded batch 1/61 (64/3862 SMILES)
decoded batch 25/61 (1600/3862 SMILES)
decoded batch 50/61 (3200/3862 SMILES)
decoded batch 61/61 (386

,dataset,n_smiles,overall_acc,string_level_acc,validity_percentage,avg_lev_dist
0,drug_like,4296,0.820104,0.415270,77.351024,4.360102
1,dyes_pigments,1728,0.658923,0.039352,50.057870,9.454861
2,natural_products,3862,0.883686,0.619109,84.904195,2.382962
3,polymers_monomers,3513,0.799835,0.465983,84.543126,2.354683
4,protein_enzyme_chemistry,2036,0.815825,0.498035,83.251473,4.157662
5,overall,15435,0.810958,0.446647,78.600583,3.952640


,dataset,input,reconstructed,valid,lev
0,drug_like,CCCCCCCCCCCCCCCCCCCCCCO,CC1CCCCCCCCCCCCCCCCCCCO,no,1
1,drug_like,CCc1ccc(CCOc2ccc(CC3SC(=O)NC3=O)cc2)nc1,CCc1ccc(CCOc2ccc(CC3=C(=O)NC3=O)cc2)nc1,no,1
2,drug_like,COc1ccc(CC(C)NCC(O)c2ccc(O)c(NC=O)c2)cc1,COc1ccc(CC(C)NCC(O)c2ccc(O)c(NC)=O)c2)cc1,no,1
3,drug_like,CN1CCN2c3ncccc3Cc3ccccc3C2C1,CN1CCN2c3ncccc3Cc3ccccc3C2)C1,no,1
4,drug_like,Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc...,Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1ncc(...,no,1
5,drug_like,C=C1C(CO)C(O)CC1n1cnc2c(=O)[nH]c(N)nc21,C=C1C(CO)C(O)CCn1cnc2c(=O)[nH]c(N)nc21,no,1
6,drug_like,CC(Nc1ncnc2nc[nH]c12)c1cc2cccc(Cl)c2c(=O)n1-c1...,CC(Nc1ncnc2nc[nH]c12)c1c2cccc(Cl)c2c(=O)n1-c1c...,no,1
7,drug_like,Nc1ccc2cc3ccc(N)cc3nc2c1,Nc1ccc2c3ccc(N)cc3nc2c1,no,1
8,drug_like,CC(=O)N=c1sc(S(N)(=O)=O)nn1C,CC(=O)Nc1sc(S(N)(=O)=O)nn1C,no,1
9,drug_like,c1cnc2cc3c(cc2n1)C1CNCC3C1,c1cnc2cc3c(cc2n1)C1CNCC2C1,no,1
